# Hyperparameter tuning with Optuna
Tuning a random forest's `n_estimators` and `max_depth` on the Pima diabetes data with Optuna's TPE sampler,
then the same study with random and grid samplers, Optuna's plots, and one study that chooses the algorithm too.

The studies are saved in `data/optuna.db`, so the figures of the Note can be drawn from them.

In [1]:
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm's notice about Jupyter widgets
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from optuna.visualization import (plot_contour, plot_optimization_history, plot_parallel_coordinate,
                                  plot_param_importances, plot_slice)

optuna.logging.set_verbosity(optuna.logging.WARNING)    # one line per trial is too much here; set INFO to see them
Path("data/optuna.db").unlink(missing_ok=True)          # start from an empty database every run
STORAGE = "sqlite:///data/optuna.db"

## 1. The data: 768 patients, 8 measurements, Outcome 1 = diabetes

In [2]:
df = pd.read_csv("data/diabetes.csv")
X = df.drop(columns="Outcome")
y = df["Outcome"]
# 70% to tune and train on, 30% kept for the final test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)

(537, 8) (231, 8)


## 2. The objective function: search space + training, returns the mean 3-fold CV accuracy

In [3]:
def objective(trial):
    # 1. the search space: the sampler picks this trial's values
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    # 2. the training
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    return cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()

## 3. A study with the TPE sampler (Bayesian optimisation), 50 trials

In [4]:
tpe = optuna.create_study(study_name="tpe", storage=STORAGE, direction="maximize",
                          sampler=optuna.samplers.TPESampler(seed=42))
tpe.optimize(objective, n_trials=50)
print("best CV accuracy:", round(tpe.best_trial.value, 3), " params:", tpe.best_trial.params)

/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campusx/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/anshu/miniforge3/envs/campus

best CV accuracy: 0.79  params: {'n_estimators': 115, 'max_depth': 8}


In [5]:
# the first trials, as Optuna would print them
tpe.trials_dataframe()[["number", "value", "params_n_estimators", "params_max_depth"]].head(8)

,number,value,params_n_estimators,params_max_depth
0,0,0.770950,106,20
1,1,0.770950,160,13
2,2,0.767225,73,5
3,3,0.770950,58,18
4,4,0.767225,140,15
5,5,0.774674,53,20
6,6,0.765363,175,6
7,7,0.756052,77,6


## 4. The final model: retrain with the best values on all training rows, test once

In [6]:
def test_best(study):
    model = RandomForestClassifier(**study.best_trial.params, random_state=42, n_jobs=-1)
    return model.fit(X_train, y_train).score(X_test, y_test)

print("test accuracy:", round(test_best(tpe), 3))

test accuracy: 0.745


In [7]:
# How noisy is 3-fold CV here? The best TPE model, scored on 10 different shuffles of the folds
from sklearn.model_selection import KFold
best_rf = RandomForestClassifier(**tpe.best_trial.params, random_state=42, n_jobs=-1)
sc = [cross_val_score(best_rf, X_train, y_train, cv=KFold(3, shuffle=True, random_state=s)).mean() for s in range(10)]
print(f"CV accuracy over 10 shuffles: mean {np.mean(sc):.3f}, std {np.std(sc):.3f}, min {min(sc):.3f}, max {max(sc):.3f}")


CV accuracy over 10 shuffles: mean 0.772, std 0.010, min 0.760, max 0.795


## 5. The same objective with a random sampler and a grid sampler

In [8]:
rand = optuna.create_study(study_name="random", storage=STORAGE, direction="maximize",
                           sampler=optuna.samplers.RandomSampler(seed=42))
rand.optimize(objective, n_trials=50)

space = {"n_estimators": [50, 100, 150, 200], "max_depth": [5, 10, 15, 20]}
grid = optuna.create_study(study_name="grid", storage=STORAGE, direction="maximize",
                           sampler=optuna.samplers.GridSampler(space, seed=42))
grid.optimize(objective, n_trials=16)        # 4 x 4 combinations

rows = []
for name, s in [("grid", grid), ("random", rand), ("TPE", tpe)]:
    v = np.array([t.value for t in s.trials])
    # how many trials landed in the region that scored best: max_depth 6-10, n_estimators up to 120
    p = s.trials_dataframe()
    region = ((p.params_max_depth.between(6, 10)) & (p.params_n_estimators <= 120)).sum()
    rows.append({"search": name, "trials": len(v), "best CV": v.max(), "mean CV": v.mean(),
                 "trials in region": region, **s.best_trial.params, "test": test_best(s)})
pd.DataFrame(rows).round(3)

,search,trials,best CV,mean CV,trials in region,n_estimators,max_depth,test
0,grid,16,0.784,0.768,2,50,15,0.749
1,random,50,0.793,0.769,7,56,8,0.740
2,TPE,50,0.790,0.771,10,115,8,0.745


## 5b. Does TPE need fewer trials? Averaged over 20 seeds
One study per sampler is a single draw of luck. Here each sampler runs 20 times (seeds 0 to 19), 50 trials each,
on a search where Bayesian optimisation is meant to pay off: an RBF SVM, whose score depends on `C` and `gamma`
over several orders of magnitude, with a good region that is a narrow band of the space.
The folds are fixed (5-fold, one shuffle), so every sampler scores a given `(C, gamma)` the same way.
We record the best score so far after each trial and average it over the 20 runs.

In [9]:
from sklearn.model_selection import StratifiedKFold
folds = StratifiedKFold(5, shuffle=True, random_state=0)

def objective_svm(trial):
    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)
    model = make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma))
    return cross_val_score(model, X_train, y_train, cv=folds).mean()

SEEDS, N = range(20), 50
best_so_far = {}
for name, make in [("random", optuna.samplers.RandomSampler), ("TPE", optuna.samplers.TPESampler)]:
    runs_ = []
    for s in SEEDS:
        st = optuna.create_study(direction="maximize", sampler=make(seed=s))
        st.optimize(objective_svm, n_trials=N)
        runs_.append(np.maximum.accumulate([t.value for t in st.trials]))
    best_so_far[name] = np.array(runs_)

curve = pd.DataFrame({"trial": np.arange(1, N + 1)})
for name, r in best_so_far.items():
    curve[f"{name}_mean"] = r.mean(axis=0)
    curve[f"{name}_se"] = r.std(axis=0, ddof=1) / np.sqrt(len(r))   # standard error of the mean
curve.to_csv("data/best_so_far.csv", index=False)
curve.set_index("trial").loc[[5, 10, 15, 20, 30, 40, 50]].round(4)


,random_mean,random_se,TPE_mean,TPE_se
trial,,,,
5,0.7616,0.0069,0.7616,0.0069
10,0.7756,0.0012,0.7756,0.0012
15,0.7764,0.0012,0.7792,0.0007
20,0.7769,0.0009,0.7797,0.0006
30,0.7781,0.0008,0.7810,0.0005
40,0.7795,0.0006,0.7818,0.0002
50,0.7804,0.0005,0.7822,0.0002


In [10]:
# how many trials random search needs to match TPE's average after 20 trials, and how often TPE is ahead
target = curve.loc[curve.trial == 20, "TPE_mean"].item()
print("TPE after 20 trials:", round(target, 4))
print("random first reaches it at trial:", curve.loc[curve.random_mean >= target, "trial"].min())
print("seeds where TPE's best after 50 trials >= random's:",
      int((best_so_far["TPE"][:, -1] >= best_so_far["random"][:, -1]).sum()), "of", len(SEEDS))


TPE after 20 trials: 0.7797
random first reaches it at trial: 42
seeds where TPE's best after 50 trials >= random's: 19 of 20


## 6. Optuna's plots (Plotly, interactive)

In [11]:
plot_optimization_history([grid, rand, tpe])

In [12]:
plot_parallel_coordinate(tpe)

In [13]:
plot_slice(tpe)

In [14]:
plot_contour(tpe)

In [15]:
plot_param_importances(tpe)

## 7. Define-by-run: the algorithm is a hyperparameter too
The search space depends on the first choice: each algorithm suggests only its own hyperparameters.

In [16]:
def make_model(trial):
    name = trial.suggest_categorical("classifier", ["SVC", "RandomForest", "GradientBoosting"])
    if name == "SVC":
        c = trial.suggest_float("C", 1e-3, 1e3, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        # SVC measures distances, so it gets scaled inputs
        model = make_pipeline(StandardScaler(), SVC(C=c, kernel=kernel, gamma="scale"))
    elif name == "RandomForest":
        model = RandomForestClassifier(n_estimators=trial.suggest_int("n_estimators", 50, 200),
                                       max_depth=trial.suggest_int("max_depth", 3, 20), random_state=42, n_jobs=-1)
    else:
        model = GradientBoostingClassifier(n_estimators=trial.suggest_int("gb_n_estimators", 50, 200),
                                           learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                           max_depth=trial.suggest_int("gb_max_depth", 2, 6), random_state=42)
    return model

def objective_any(trial):
    return cross_val_score(make_model(trial), X_train, y_train, cv=3).mean()

anyalgo = optuna.create_study(study_name="algorithm", storage=STORAGE, direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=42))
anyalgo.optimize(objective_any, n_trials=100)
print("best CV accuracy:", round(anyalgo.best_trial.value, 3), anyalgo.best_trial.params)

best CV accuracy: 0.788 {'classifier': 'RandomForest', 'n_estimators': 134, 'max_depth': 8}


In [17]:
runs = anyalgo.trials_dataframe()
# how often TPE tried each algorithm, and how well it did
runs.groupby("params_classifier")["value"].agg(["count", "mean", "max"]).round(3)

,count,mean,max
params_classifier,,,
GradientBoosting,36,0.765,0.780
RandomForest,51,0.775,0.788
SVC,13,0.735,0.780


In [18]:
# The SVC trials: score against C and kernel; and the share of the majority class in training
print(runs[runs["params_classifier"] == "SVC"][["value", "params_C", "params_kernel"]].sort_values("value"))
print("majority-class share:", round(1 - y_train.mean(), 3))

       value    params_C params_kernel
2   0.649907    0.012330           rbf
49  0.649907    0.001112           rbf
6   0.694600    0.003855        linear
45  0.711359   70.955130           rbf
43  0.724395    0.121351           rbf
41  0.743017    0.141932           rbf
42  0.756052    6.204885           rbf
8   0.763501    1.906609           rbf
3   0.765363    4.689401           rbf
33  0.770950  966.766656        linear
44  0.772812   27.283295        linear
38  0.772812  794.817365        linear
40  0.780261    0.620222           rbf
majority-class share: 0.65


In [19]:
# when each algorithm was tried: trial numbers in blocks of 20
runs["block"] = runs["number"] // 20 * 20
pd.crosstab(runs["block"], runs["params_classifier"])

params_classifier,GradientBoosting,RandomForest,SVC
block,,,
0,8,8,4
20,17,1,2
40,11,2,7
60,0,20,0
80,0,20,0


In [20]:
# the best model, rebuilt from the best values: FixedTrial answers every suggest_ call from a dictionary
best_model = make_model(optuna.trial.FixedTrial(anyalgo.best_trial.params))
print(best_model)
print("test accuracy:", round(best_model.fit(X_train, y_train).score(X_test, y_test), 3))

RandomForestClassifier(max_depth=8, n_estimators=134, n_jobs=-1,
                       random_state=42)


test accuracy: 0.749
